## **Business Understanding**

### Real-World Problem
SyriaTel has been losing immense revenue as a result of customer churn. It is in the best interest of any business to retain its existing customers. For this reason, the telecom is incurring marketing costs in a bid to reach new customer base. It is a zero sum game as a significant number of the acquired customers end up churning as well. The task at hand is to determine customers likely to churn by looking out for patterns and come up with measures that would prevent that from happening.

### Stakeholders

1. **Executive Management (CEO, COO, Strategy Team):** This stakeholder group will use this project to quantify expected revenue loss. They will then be able to track churn trends over time by customer segment. That way they can make strategic decisions on whether to invest more in retention programs or acquisition.
2. **Marketing Team:** The marketing team will use this project to target high-risk customers with personalized retention offers. This will reduce spending on mass acquisition campaigns. Additionally, the marketing team can run data-driven retention campaigns instead of blanket promotions.
3. **Customer Service Team:** The project will enable this stakeholder group to flag high-risk customers in the CRM system.This make it possible to prioritize support for customers likely to churn. They will also provide proactive outreach before customers cancel.
4. **Data & Analytics Team:** This group will be interested in monitoring model accuracy, recall, and drift.They might also retrain the model with new data and identify new churn drivers as customer behavior changes.

### Business Questions
1. Which customers are most likely to churn?
2. What factors contribute most to churn?
3. How well can we predict churn compared to random guessing?
4. What actions can SyriaTel take based on these predictions?






In [6]:
#importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier


In [7]:
#loading the dataset
data =pd.read_csv("data\Telecom.csv")
data.head()

,state,account length,area code,phone number,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,...,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
0,KS,128,415,382-4657,no,yes,25,265.1,110,45.07,...,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,371-7191,no,yes,26,161.6,123,27.47,...,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,358-1921,no,no,0,243.4,114,41.38,...,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,375-9999,yes,no,0,299.4,71,50.90,...,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,330-6626,yes,no,0,166.7,113,28.34,...,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [8]:
#Summary of dataframe
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   state                   3333 non-null   object 
 1   account length          3333 non-null   int64  
 2   area code               3333 non-null   int64  
 3   phone number            3333 non-null   object 
 4   international plan      3333 non-null   object 
 5   voice mail plan         3333 non-null   object 
 6   number vmail messages   3333 non-null   int64  
 7   total day minutes       3333 non-null   float64
 8   total day calls         3333 non-null   int64  
 9   total day charge        3333 non-null   float64
 10  total eve minutes       3333 non-null   float64
 11  total eve calls         3333 non-null   int64  
 12  total eve charge        3333 non-null   float64
 13  total night minutes     3333 non-null   float64
 14  total night calls       3333 non-null   

In [9]:
#Checking for missing values
data.isna().sum()

state                     0
account length            0
area code                 0
phone number              0
international plan        0
voice mail plan           0
number vmail messages     0
total day minutes         0
total day calls           0
total day charge          0
total eve minutes         0
total eve calls           0
total eve charge          0
total night minutes       0
total night calls         0
total night charge        0
total intl minutes        0
total intl calls          0
total intl charge         0
customer service calls    0
churn                     0
dtype: int64

In [10]:
#Checking for duplicate values in the dataframe
data.duplicated().sum()

0

## **Data Understanding**

### Data Source
The data source is Kaggle and is titled *Churn in Telecom's dataset.* The owner of the dataset is David Becks.
It consists of account-level records for SyriaTel subscribers, containing structured features that describe customer service plans, call usage behavior, and support interactions. Each row represents a unique customer account, and the associated features are used as predictors in a supervised learning framework to model and classify customer churn outcomes.

### Suitability of dataset
1. Contains customer-level data relevant to churn behavior
2. Includes usage, service, and support interaction features that influence churn
3. Has a clearly defined churn target variable
4. Suitable for supervised machine learning classification
5. Enables identification of patterns and drivers of customer attrition

### Dataset Overview

- The dataset contains **3,333 customer records** and **21 features (20 predictors + 1 target)**.
- Each record represents a **unique SyriaTel customer account**.
- The features include a mix of **numerical variables** (such as call minutes, call counts, charges, account length, and customer service calls) and **categorical variables** (including state, area code, international plan, and voicemail plan).
- The target variable is **churn**, a binary indicator showing whether a customer discontinued the service.
- Descriptive statistics were generated for all features to understand data distributions, detect potential outliers, and inform preprocessing decisions.

### Feature Selection Justification

The features included in this analysis were selected based on their relevance to customer churn and their suitability for predictive modeling. Usage-related variables such as call minutes, call counts, and associated charges capture customer engagement and consumption patterns, which are strong indicators of satisfaction and likelihood to remain with the service. Service plan attributes, including international and voicemail plans, provide insight into customer needs and pricing sensitivity, both of which influence churn behavior. Customer service interaction features, particularly the number of customer service calls, reflect service quality and customer experience, which are commonly linked to customer attrition. Account-level attributes such as account length represent customer tenure and loyalty. Together, these features are well-defined, largely structured, and measurable prior to churn, making them appropriate predictors for a supervised churn classification model.

### Data Limitations

The dataset is historical and observational, meaning it captures past customer behavior but cannot establish causal relationships between features and churn:
1. The churn variable is imbalanced, with fewer churned customers than non-churned customers, which may bias models toward the majority class if not properly addressed. 
2. Additionally, the data lacks demographic, billing, and customer satisfaction information, which could further improve churn prediction. 
3. Temporal information is limited, preventing analysis of behavior changes over time. 
4. Finally, the dataset represents a snapshot of customer behavior and may not fully reflect future trends or external factors influencing churn.


## **Data Preparation**

### Overview
